# Notebook 7: Observability, Governance & Migration — Production-Grade Agent Operations
## Tracing, Monitoring, Evaluation, M365 Integration, Limits, Permissions & Migration Paths

**Sources:**
- [Agent Tracing Concepts](https://learn.microsoft.com/en-us/azure/foundry/observability/concepts/trace-agent-concept)
- [Trace Setup](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/trace-agent-setup)
- [Client-Side Tracing](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/trace-agent-client-side?tabs=python)
- [Framework Tracing](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/trace-agent-framework)
- [Prompt Optimizer](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/prompt-optimizer)
- [Monitoring Dashboard](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/how-to-monitor-agents-dashboard?tabs=python)
- [Agent Evaluation](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/evaluate-agent)
- [Agent 365 Integration](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/agent-365-integration)
- [Migration Guide](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/migrate?tabs=python)
- [Hosted Agent Preview Migration](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/migrate-hosted-agent-preview)
- [Agent Applications Migration](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/migrate-agent-applications)
- [Limits, Quotas & Regions](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/limits-quotas-regions)
- [Hosted Agent Permissions](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agent-permissions)
- [FAQ](https://learn.microsoft.com/en-us/azure/foundry/agents/faq)
- [Capability Hosts](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/capability-hosts)
- [MCP Tool Governance](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/governance)

## Master Architecture Diagram

```
┌───────────────────────────────────────────────────────────────────────────────────────────────┐
│            MICROSOFT FOUNDRY — OBSERVABILITY, GOVERNANCE & MIGRATION LANDSCAPE                │
│                                                                                               │
│  ┌──────────── OBSERVABILITY STACK ──────────────┐  ┌────────── GOVERNANCE LAYER ──────────┐  │
│  │                                                │  │                                      │  │
│  │  Agent Code ──► OpenTelemetry SDK              │  │  Agent 365 (Enterprise Control)      │  │
│  │       │              │                         │  │  ├── Registry (agent inventory)      │  │
│  │       │         Trace Exporter                 │  │  ├── Access Control (Entra ID)       │  │
│  │       │              │                         │  │  ├── Visualization (real-time)       │  │
│  │       ▼              ▼                         │  │  ├── Interoperability (M365 apps)    │  │
│  │  Azure Monitor Application Insights            │  │  └── Security (Defender/Purview)     │  │
│  │       │                                        │  │                                      │  │
│  │       ├── Traces (spans, attributes)           │  │  AI Gateway (API Management)         │  │
│  │       ├── Conversations (history, steps)       │  │  ├── Rate limiting                   │  │
│  │       ├── Metrics (tokens, latency)            │  │  ├── IP filtering                    │  │
│  │       └── Evaluations (quality, safety)        │  │  ├── Audit logging                   │  │
│  │                                                │  │  └── Routing policies                │  │
│  └────────────────────────────────────────────────┘  └──────────────────────────────────────┘  │
│                                                                                               │
│  ┌──────────── FRAMEWORK INTEGRATIONS ───────────┐  ┌──────── CAPABILITY HOSTS ─────────────┐ │
│  │                                                │  │                                       │ │
│  │  Microsoft Agent Framework (native)            │  │  Account-Level ──► Project-Level      │ │
│  │  LangChain / LangGraph (langchain-azure-ai)   │  │       │                  │            │ │
│  │  OpenAI Agents SDK (OTel instrumentor)         │  │       ▼                  ▼            │ │
│  │  Semantic Kernel (auto-instrumentation)        │  │  Azure Cosmos DB  (threads)           │ │
│  │                                                │  │  Azure Storage    (files)             │ │
│  └────────────────────────────────────────────────┘  │  Azure AI Search  (vector stores)     │ │
│                                                      └───────────────────────────────────────┘ │
│                                                                                               │
│  ┌──────────── MIGRATION PATHS ──────────────────────────────────────────────────────────────┐│
│  │                                                                                           ││
│  │  Assistants API ──► Foundry Agent Service (threads→conversations, runs→responses)         ││
│  │  Hosted Preview  ──► Refreshed Preview (new protocols, session-based sandbox)             ││
│  │  Agent Applications ──► Agent Object Model (unified identity, stable endpoints)           ││
│  │                                                                                           ││
│  └───────────────────────────────────────────────────────────────────────────────────────────┘│
└───────────────────────────────────────────────────────────────────────────────────────────────┘
```

---
## Prerequisites

| Requirement | Description | Status |
|---|---|---|
| **Azure Subscription** | Active subscription | [ ] |
| **Foundry Project** | Created in Notebook 03 | [ ] |
| **Python 3.10+** | Runtime | [ ] |
| **Packages** | `azure-ai-projects>=2.0.0`, `azure-identity`, `opentelemetry-sdk`, `azure-monitor-opentelemetry`, `azure-core-tracing-opentelemetry` | [ ] |
| **Model Deployed** | gpt-4o or gpt-4o-mini | [ ] |
| **Application Insights** | Connected to Foundry project | [ ] |
| **Log Analytics Reader** | Role assigned for viewing traces | [ ] |

In [ ]:
# Install required packages
# !pip install azure-ai-projects azure-identity opentelemetry-sdk azure-core-tracing-opentelemetry azure-monitor-opentelemetry python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Core configuration
FOUNDRY_PROJECT_ENDPOINT = os.environ.get("FOUNDRY_PROJECT_ENDPOINT", "your-endpoint-here")
FOUNDRY_MODEL_NAME = os.environ.get("FOUNDRY_MODEL_NAME", "gpt-4o-mini")

print(f"Project Endpoint: {FOUNDRY_PROJECT_ENDPOINT[:50]}...")
print(f"Model: {FOUNDRY_MODEL_NAME}")

---
# PART 1: AGENT TRACING & OBSERVABILITY
---

## 1.1 Agent Tracing — Key Concepts

Microsoft Foundry provides an observability platform built on **OpenTelemetry** for monitoring and tracing AI agents. Tracing captures key details during an agent run:

- **User inputs and agent outputs**
- **Tool usage** (tool calls and results)
- **Token consumption**
- **Time signals** (duration and latency)

### Why Tracing Matters
Complex agents present challenges:
- High number of steps involved in generating a response
- Step sequences vary based on user input
- Inputs/outputs at each stage may be long
- Nested execution (agent → tool → sub-process → tool)

### Core Concepts

| Concept | Description |
|---|---|
| **Traces** | Capture the journey of a request through your application by recording events and state changes |
| **Spans** | Building blocks of traces — represent single operations with start/end times, can be nested |
| **Attributes** | Key-value pairs attached to spans providing contextual metadata |
| **Semantic Conventions** | OpenTelemetry standardized names/formats for trace data attributes |
| **Trace Exporters** | Send trace data to backend systems (Application Insights, console, OTLP) |

### Multi-Agent Observability (OpenTelemetry Extensions)

Microsoft + Cisco Outshift introduced new semantic conventions for multi-agent systems:

| Type | Context | Name/Attribute | Purpose |
|---|---|---|---|
| Span | — | `execute_task` | Task planning and event propagation |
| Child Span | `invoke_agent` | `agent_to_agent_interaction` | Traces inter-agent communication |
| Child Span | `invoke_agent` | `agent.state.management` | Memory management |
| Child Span | `invoke_agent` | `agent_planning` | Agent's internal planning steps |
| Child Span | `invoke_agent` | `agent orchestration` | Agent-to-agent orchestration |
| Attribute | `invoke_agent` | `tool_definitions` | Tool's purpose/configuration |
| Attribute | `execute_tool` | `tool.call.arguments` | Arguments passed to tool |
| Attribute | `execute_tool` | `tool.call.results` | Results returned by tool |
| Event | — | Evaluation | Structured evaluation of agent performance |

These conventions are integrated into: Foundry, Microsoft Agent Framework, LangChain, LangGraph, and OpenAI Agents SDK.

> **Note:** Tracing is generally available for prompt agents only. Workflow, hosted, and custom agents are in preview.

## 1.2 Setting Up Tracing

### Step 1: Connect Application Insights to Your Foundry Project

1. Sign in to [Microsoft Foundry](https://ai.azure.com) → ensure **New Foundry** toggle is on
2. Open your Foundry project
3. In the left navigation, select **Agents**
4. At the top, select **Traces**
5. On the right, select **Connect** to create/connect an Application Insights resource

### Required RBAC
- **Log Analytics Reader** role — for viewing traces, insights, and visualizations
- Assign via Azure portal → Access control (IAM)

### Instrumentation Approaches

| Approach | Description | Code Changes? |
|---|---|---|
| **Server-side traces** | Automatic for Prompt/Host agents and workflows | None |
| **Client-side traces** | Python/C# SDK with OpenTelemetry | Yes |
| **VS Code local tracing** | Foundry Toolkit with local OTLP collector | Minimal |

## 1.3 Client-Side Tracing — Full Python Example

In [ ]:
# ============================================================
# Client-Side Tracing with Azure Monitor Export
# ============================================================

import os

# IMPORTANT: Set BEFORE importing instrumentation
os.environ["AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING"] = "true"

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import DefaultAzureCredential
from azure.monitor.opentelemetry import configure_azure_monitor
from opentelemetry import trace

endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"]

with (
    DefaultAzureCredential() as credential,
    AIProjectClient(endpoint=endpoint, credential=credential) as project,
):
    # Get the Application Insights connection string from the project
    connection_string = project.telemetry.get_application_insights_connection_string()
    configure_azure_monitor(connection_string=connection_string)

    tracer = trace.get_tracer(__name__)

    with tracer.start_as_current_span("agent-tracing-scenario"):
        with project.get_openai_client() as openai:
            # Create an agent
            agent = project.agents.create_version(
                agent_name="MyTracedAgent",
                definition=PromptAgentDefinition(
                    model=os.environ["FOUNDRY_MODEL_NAME"],
                    instructions="You are a helpful assistant.",
                ),
            )
            print(f"Agent created (id: {agent.id}, name: {agent.name})")

            # Create a conversation and get a response
            conversation = openai.conversations.create()
            response = openai.responses.create(
                conversation=conversation.id,
                extra_body={"agent_reference": {
                    "name": agent.name, "id": agent.id, "type": "agent_reference"
                }},
                input="What is the largest city in France?",
            )
            print(f"Response: {response.output_text}")

            # Clean up
            openai.conversations.delete(conversation_id=conversation.id)
            project.agents.delete_version(
                agent_name=agent.name, agent_version=agent.version
            )

## 1.4 Console Export (Local Debugging)

In [ ]:
# ============================================================
# Console Export — useful for local debugging
# ============================================================

import os
os.environ["AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING"] = "true"

from azure.ai.projects.telemetry import AIProjectInstrumentor
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

# Set up console tracing
tracer_provider = TracerProvider()
tracer_provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(tracer_provider)

# Enable instrumentation
AIProjectInstrumentor().instrument()

tracer = trace.get_tracer(__name__)
print("Console tracing configured — spans will print to stdout")

## 1.5 Tracing Custom Functions

In [ ]:
# ============================================================
# Custom Function Tracing with @trace_function decorator
# ============================================================

from azure.ai.projects.telemetry import trace_function

@trace_function
def fetch_weather(location: str) -> str:
    """Get the current weather for a location."""
    return f"Weather in {location}: sunny, 72°F"

# With custom span name
@trace_function("get-current-weather")
def fetch_weather_named(location: str) -> str:
    """Get the current weather for a location."""
    return f"Weather in {location}: sunny, 72°F"

# The decorator records:
# - Parameters as code.function.parameter.<name> span attributes
# - Return values as code.function.return.value
# - Supported types: str, int, float, bool, list, dict, tuple, set

# NOTE: @trace_function ALWAYS traces parameters and return values
# regardless of OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT setting

print(fetch_weather("Seattle"))
print(fetch_weather_named("Paris"))

## 1.6 Custom Span Attributes

In [ ]:
# ============================================================
# Custom SpanProcessor — inject metadata into every span
# ============================================================

from opentelemetry.sdk.trace import SpanProcessor, ReadableSpan
from opentelemetry.trace import Span

class CustomAttributeSpanProcessor(SpanProcessor):
    def on_start(self, span: Span, parent_context=None):
        span.set_attribute("session.id", "user-session-abc")
        span.set_attribute("environment", "production")

    def on_end(self, span: ReadableSpan):
        pass

# Register with the global tracer provider:
# from typing import cast
# from opentelemetry import trace
# from opentelemetry.sdk.trace import TracerProvider
# provider = cast(TracerProvider, trace.get_tracer_provider())
# provider.add_span_processor(CustomAttributeSpanProcessor())

print("Custom span processor defined")

## 1.7 Tracing Environment Variables Reference

| Variable | Default | Description |
|---|---|---|
| `AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING` | `false` | Enable GenAI tracing instrumentation |
| `OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT` | `false` | Capture message contents and tool call parameters |
| `AZURE_TRACING_GEN_AI_ENABLE_TRACE_CONTEXT_PROPAGATION` | `true`* | Inject W3C Trace Context headers |
| `AZURE_TRACING_GEN_AI_TRACE_CONTEXT_PROPAGATION_INCLUDE_BAGGAGE` | `false` | Include baggage header |
| `AZURE_TRACING_GEN_AI_INSTRUMENT_RESPONSES_API` | `true` | Auto-instrument Responses/Conversations APIs |
| `AZURE_TRACING_GEN_AI_INCLUDE_BINARY_DATA` | `false` | Include image/file data in spans |

\* Default is `true` when tracing is enabled.

### Programmatic Configuration (Alternative to env vars)
```python
from azure.ai.projects.telemetry import AIProjectInstrumentor

AIProjectInstrumentor().instrument(
    enable_content_recording=True,
    enable_trace_context_propagation=True,
    enable_baggage_propagation=False,
)
```

> **Security Warning:** Content recording captures user messages, tool call arguments, and model outputs. Only enable in development!

---
# PART 2: FRAMEWORK TRACING INTEGRATIONS
---

## 2.1 Microsoft Agent Framework

**Zero-code tracing** — agents automatically emit traces when tracing is enabled for your project.

1. Run your agent at least once
2. In Foundry portal → **Observability** → **Traces**
3. Confirm new traces appear (2-5 minutes after execution)

## 2.2 LangChain / LangGraph Tracing

In [ ]:
# ============================================================
# LangGraph Agent with Azure AI Tracing
# ============================================================
# pip install langchain-azure-ai langgraph langchain langchain-openai azure-identity python-dotenv

import os
from dotenv import load_dotenv
from langchain_azure_ai.callbacks.tracers import AzureAIOpenTelemetryTracer

load_dotenv(override=True)

# Initialize the tracer
azure_tracer = AzureAIOpenTelemetryTracer(
    connection_string=os.environ.get("APPLICATION_INSIGHTS_CONNECTION_STRING"),
    enable_content_recording=os.getenv("OTEL_RECORD_CONTENT", "true").lower() == "true",
    name="Music Player Agent",
)

# Define tools
from langchain_core.tools import tool

@tool
def play_song_on_spotify(song: str):
    """Play a song on Spotify"""
    return f"Successfully played {song} on Spotify!"

@tool
def play_song_on_apple(song: str):
    """Play a song on Apple Music"""
    return f"Successfully played {song} on Apple Music!"

tools = [play_song_on_apple, play_song_on_spotify]

# Model setup with Azure OpenAI
import azure.identity
from langchain_openai import AzureChatOpenAI

token_provider = azure.identity.get_bearer_token_provider(
    azure.identity.DefaultAzureCredential(),
    "https://ai.azure.com/.default",
)

model = AzureChatOpenAI(
    azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
    azure_deployment=os.environ.get("AZURE_OPENAI_CHAT_DEPLOYMENT"),
    openai_api_version=os.environ.get("AZURE_OPENAI_VERSION"),
    azure_ad_token_provider=token_provider,
).bind_tools(tools, parallel_tool_calls=False)

# Build the LangGraph workflow
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

tool_node = ToolNode(tools)

def should_continue(state: MessagesState):
    messages = state["messages"]
    last_message = messages[-1]
    return "continue" if getattr(last_message, "tool_calls", None) else "end"

def call_model(state: MessagesState):
    messages = state["messages"]
    response = model.invoke(messages)
    return {"messages": [response]}

workflow = StateGraph(MessagesState)
workflow.add_node("agent", call_model)
workflow.add_node("action", tool_node)
workflow.add_edge(START, "agent")
workflow.add_conditional_edges(
    "agent", should_continue, {"continue": "action", "end": END}
)
workflow.add_edge("action", "agent")

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

# Run with tracing
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}, "callbacks": [azure_tracer]}
input_message = HumanMessage(content="Can you play Taylor Swift's most popular song?")

for event in app.stream({"messages": [input_message]}, config, stream_mode="values"):
    event["messages"][-1].pretty_print()

## 2.3 OpenAI Agents SDK Tracing

In [ ]:
# ============================================================
# OpenAI Agents SDK with OpenTelemetry
# ============================================================
# pip install opentelemetry-sdk opentelemetry-instrumentation-openai-agents azure-monitor-opentelemetry-exporter

import os
from opentelemetry import trace
from opentelemetry.instrumentation.openai_agents import OpenAIAgentsInstrumentor
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor, ConsoleSpanExporter

# Configure tracer provider + exporter
resource = Resource.create({
    "service.name": os.getenv("OTEL_SERVICE_NAME", "openai-agents-app"),
})
provider = TracerProvider(resource=resource)

conn = os.getenv("APPLICATION_INSIGHTS_CONNECTION_STRING")
if conn:
    from azure.monitor.opentelemetry.exporter import AzureMonitorTraceExporter
    provider.add_span_processor(
        BatchSpanProcessor(AzureMonitorTraceExporter.from_connection_string(conn))
    )
else:
    provider.add_span_processor(BatchSpanProcessor(ConsoleSpanExporter()))

trace.set_tracer_provider(provider)

# Instrument the OpenAI Agents SDK
OpenAIAgentsInstrumentor().instrument(tracer_provider=trace.get_tracer_provider())

# Create a session span around your agent run
tracer = trace.get_tracer(__name__)
with tracer.start_as_current_span("agent_session[openai.agents]"):
    # ... run your agent here
    print("OpenAI Agents SDK instrumented and ready")

---
# PART 3: PROMPT OPTIMIZER
---

## 3.1 Prompt Optimizer (Preview)

Prompt Optimizer automatically improves agent system instructions using AI-driven prompt engineering.

### How It Works
1. **Input Collection** — provide initial description or open with existing instructions
2. **LLM-Based Optimization** — restructures, clarifies, and enhances instructions
3. **Reasoning Generation** — per-paragraph explanations of changes
4. **Iterative Refinement** — add suggestions and re-optimize

### How to Access
1. Foundry portal → **Build** → **Agents** → select agent
2. Find the **Instructions** section
3. Click the pencil-with-sparkle icon next to *Instructions*

### Supported Regions
Central US, East US 2, France Central, Germany West Central, Italy North, Japan West, North Central US, Poland Central, Spain Central, Sweden Central, Switzerland West, UAE North, West US, West US 2, West US 3

### Best Practices
- **Start simple, then refine** — let the optimizer create initial structure
- **Use specific suggestions** — "add error handling for invalid dates" beats "make it better"
- **Review reasoning** — catch changes that don't align with your use case
- **Test after optimizing** — verify in playground before deploying
- **Run a full evaluation** — measure whether changes actually improve performance

> **Limitation:** Text-based instructions only. Results aren't persisted — click **Use prompt** before closing!

---
# PART 4: AGENT MONITORING DASHBOARD
---

## 4.1 Dashboard Overview

Access: Foundry portal → **Build** → select agent → **Monitor** tab

### Key Metrics

| Metric | Description | Alert Threshold |
|---|---|---|
| **Token Usage** | Token counts for agent traffic | High usage → verbose prompts |
| **Latency** | Response time for agent runs | >10s → throttling/complex tools |
| **Run Success Rate** | % of runs completing successfully | <95% → investigate failures |
| **Evaluation Metrics** | Scores from evaluators on sampled outputs | Varies by evaluator |
| **Red Teaming Results** | Outcomes from adversarial scans | Failed → security risks |

### Monitor Settings

| Setting | Purpose |
|---|---|
| **Continuous Evaluation** | Run evaluators on sampled responses (set sample rate, add evaluators) |
| **Scheduled Evaluations** (preview) | Run on a schedule against benchmarks |
| **Red Team Scans** (preview) | Adversarial tests for data leakage, prohibited actions |
| **Alerts** (preview) | Detect anomalies in latency, tokens, eval scores |

## 4.2 Setting Up Continuous Evaluation

In [ ]:
# ============================================================
# Continuous Evaluation Setup
# ============================================================

import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
    EvaluationRule,
    ContinuousEvaluationRuleAction,
    EvaluationRuleFilter,
    EvaluationRuleEventType,
)

load_dotenv()
endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]

with (
    DefaultAzureCredential() as credential,
    AIProjectClient(endpoint=endpoint, credential=credential) as project_client,
    project_client.get_openai_client() as openai_client,
):
    # Create an agent
    agent = project_client.agents.create_version(
        agent_name=os.environ["AZURE_AI_AGENT_NAME"],
        definition=PromptAgentDefinition(
            model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
            instructions="You are a helpful assistant that answers general questions",
        ),
    )
    print(f"Agent created (id: {agent.id}, name: {agent.name})")

    # Create evaluation with violence detection
    data_source_config = {"type": "azure_ai_source", "scenario": "responses"}
    testing_criteria = [
        {
            "type": "azure_ai_evaluator",
            "name": "violence_detection",
            "evaluator_name": "builtin.violence",
        }
    ]
    eval_object = openai_client.evals.create(
        name="Continuous Evaluation",
        data_source_config=data_source_config,
        testing_criteria=testing_criteria,
    )
    print(f"Evaluation created (id: {eval_object.id})")

    # Create the continuous evaluation rule
    continuous_eval_rule = project_client.evaluation_rules.create_or_update(
        id="my-continuous-eval-rule",
        evaluation_rule=EvaluationRule(
            display_name="My Continuous Eval Rule",
            description="Runs on agent response completions",
            action=ContinuousEvaluationRuleAction(
                eval_id=eval_object.id, max_hourly_runs=100
            ),
            event_type=EvaluationRuleEventType.RESPONSE_COMPLETED,
            filter=EvaluationRuleFilter(agent_name=agent.name),
            enabled=True,
        ),
    )
    print(f"Rule created (id: {continuous_eval_rule.id})")

---
# PART 5: AGENT EVALUATION
---

## 5.1 Evaluation Concepts

### Built-in Evaluator Categories

| Category | Examples |
|---|---|
| **Agent Evaluators** | Task adherence, tool usage, user intent handling |
| **Quality Evaluators** | Coherence, fluency, relevance |
| **Text Similarity** | NLP metrics against reference answers |
| **Safety Evaluators** | Violence, self-harm, sexual, hate content detection |

### Evaluation Workflow
1. Set up SDK client
2. Choose evaluators
3. Create test dataset (JSONL with `query` field)
4. Upload dataset
5. Run evaluation
6. Interpret results

In [ ]:
# ============================================================
# Agent Evaluation — Full Example
# ============================================================

import os
import time
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_deployment = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
client = project_client.get_openai_client()

# Define evaluators
testing_criteria = [
    {
        "type": "azure_ai_evaluator",
        "name": "Task Adherence",
        "evaluator_name": "builtin.task_adherence",
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{sample.output_items}}",
        },
        "initialization_parameters": {"deployment_name": model_deployment},
    },
    {
        "type": "azure_ai_evaluator",
        "name": "Coherence",
        "evaluator_name": "builtin.coherence",
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{sample.output_text}}",
        },
        "initialization_parameters": {"deployment_name": model_deployment},
    },
    {
        "type": "azure_ai_evaluator",
        "name": "Violence",
        "evaluator_name": "builtin.violence",
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{sample.output_text}}",
        },
    },
]

# Create evaluation
data_source_config = {
    "type": "custom",
    "item_schema": {
        "type": "object",
        "properties": {"query": {"type": "string"}},
        "required": ["query"],
    },
    "include_sample_schema": True,
}

evaluation = client.evals.create(
    name="Agent Quality Evaluation",
    data_source_config=data_source_config,
    testing_criteria=testing_criteria,
)
print(f"Evaluation created: {evaluation.id}")

# Upload test dataset (create test-queries.jsonl first)
# dataset = project_client.datasets.upload_file(
#     name="agent-test-queries", version="1", file_path="./test-queries.jsonl"
# )

# Create evaluation run
# eval_run = client.evals.runs.create(
#     eval_id=evaluation.id,
#     name="Agent Evaluation Run",
#     data_source={
#         "type": "azure_ai_target_completions",
#         "source": {"type": "file_id", "id": dataset.id},
#         "input_messages": {
#             "type": "template",
#             "template": [{"type": "message", "role": "user",
#                           "content": {"type": "input_text", "text": "{{item.query}}"}}],
#         },
#         "target": {"type": "azure_ai_agent", "name": "my-agent", "version": "1"},
#     },
# )

# Poll for results
# while True:
#     run = client.evals.runs.retrieve(run_id=eval_run.id, eval_id=evaluation.id)
#     if run.status in ["completed", "failed"]:
#         break
#     time.sleep(5)
# print(f"Status: {run.status}")
# print(f"Report URL: {run.report_url}")

---
# PART 6: MICROSOFT AGENT 365 INTEGRATION
---

## 6.1 Agent 365 — Enterprise Control Plane

Microsoft Agent 365 provides a single place to observe, govern, and secure every agent across an organization.

### Five Pillars

| Pillar | Description |
|---|---|
| **Registry** | Complete inventory of all agents (Foundry, Copilot Studio, admin-registered, shadow agents) |
| **Access Control** | Entra ID-based controls and risk-based Conditional Access policies |
| **Visualization** | Explore connections between agents, people, and data in real-time |
| **Interoperability** | Agents access M365 apps and organizational data + Work IQ |
| **Security** | Microsoft Defender + Purview integration for threat/vulnerability protection |

### How Foundry Integrates with Agent 365

1. **Automatic Registry Sync** — Published Foundry agents appear in the Agent 365 registry
2. **Digital Worker Publishing** — Hosted agents published as *digital workers* with their own Entra Agent ID

### Supported Agent Types

| Agent Type | Registry Sync | Digital Worker | Activity Data |
|---|---|---|---|
| **Prompt Agent** | Yes | Yes | Yes |
| **Hosted Agent** | Yes | Yes | Via A365 SDK |
| **Workflow Agent** | Yes | No | No |

### Enablement Requirements
1. Microsoft 365 Copilot license + [Frontier preview program](https://adoption.microsoft.com/copilot/frontier-program/)
2. Global administrator enables Agent 365 and accepts terms in [M365 admin center](https://admin.microsoft.com/)

### Data Residency

| Platform | Residency Model |
|---|---|
| **Microsoft Foundry** | Azure region selected at resource creation |
| **Microsoft Agent 365** | Storage location of the Entra tenant |

> When data flows from Foundry → Agent 365, it moves from region-based to tenant-based residency.

---
# PART 7: CAPABILITY HOSTS
---

## 7.1 Understanding Capability Hosts

Capability hosts tell Foundry Agent Service where to store and process agent data:
- **Conversation history** (threads)
- **File uploads**
- **Vector stores**

### Default vs. Bring-Your-Own

| Setup | Storage | Use Case |
|---|---|---|
| **Basic** (no capability host) | Microsoft-managed resources | Dev/test |
| **Standard** (capability hosts) | Your own Azure resources | Production/compliance |

### Configuration Hierarchy
1. **Service defaults** (Microsoft-managed) — when no capability host configured
2. **Account-level** — shared defaults for all projects
3. **Project-level** — overrides account-level for specific project

### Resource Connections

| Property | Azure Resource | Purpose |
|---|---|---|
| `threadStorageConnections` | Azure Cosmos DB | Agent definitions + conversation history |
| `vectorStoreConnections` | Azure AI Search | Vector storage for retrieval |
| `storageConnections` | Azure Storage Account | File uploads and blob storage |
| `aiServicesConnections` (optional) | Azure OpenAI | Use your own model deployments |

In [ ]:
# ============================================================
# Capability Host Configuration (REST API)
# ============================================================

# Account-level capability host
account_capability_host = """
PUT https://management.azure.com/subscriptions/{subscriptionId}/resourceGroups/{resourceGroupName}/
    providers/Microsoft.CognitiveServices/accounts/{accountName}/
    capabilityHosts/{name}?api-version=2025-06-01

{
  "properties": {
    "capabilityHostKind": "Agents"
  }
}
"""

# Project-level capability host
project_capability_host = """
PUT https://management.azure.com/subscriptions/{subscriptionId}/resourceGroups/{resourceGroupName}/
    providers/Microsoft.CognitiveServices/accounts/{accountName}/
    projects/{projectName}/capabilityHosts/{name}?api-version=2025-06-01

{
  "properties": {
    "capabilityHostKind": "Agents",
    "threadStorageConnections": ["my-cosmos-db-connection"],
    "vectorStoreConnections": ["my-ai-search-connection"],
    "storageConnections": ["my-storage-account-connection"],
    "aiServicesConnections": ["my-azure-openai-connection"]
  }
}
"""

print("Capability host REST API templates defined")
print("\nKey constraints:")
print("  - One capability host per scope (account/project)")
print("  - Cannot update — must delete and recreate")
print("  - Account capability host required before project-level")

---
# PART 8: QUOTAS, LIMITS & REGIONS
---

## 8.1 Service Limits

| Limit | Value |
|---|---|
| Max files per agent/thread | **10,000** |
| Max file size for agents | **512 MB** |
| Max total uploaded files | **300 GB** |
| Max file size for vector store attachment | **2,000,000 tokens** |
| Max messages per thread | **100,000** |
| Max text content per message | **1,500,000 characters** |
| Max tools per agent | **128** |

> These limits are **fixed** and apply uniformly. Rate limiting is at the model deployment level.

### Limit Error Reference

| Scenario | HTTP | Code | Action |
|---|---|---|---|
| File too large | 400 | `file_size_exceeded` | Split into smaller files |
| Vector store token limit | 400 | `token_limit_exceeded` | Reduce content or split |
| Thread message cap | 400 | `message_limit_exceeded` | Create new thread |
| Message content too large | 400 | `content_size_exceeded` | Use file search |
| Too many tools | 400 | `tool_limit_exceeded` | Remove unused tools |
| Rate limit | 429 | `rate_limit_exceeded` | Exponential backoff |

### Supported Foundry Models

| Model | Specialty |
|---|---|
| MAI-DS-R1 | Deterministic, precision-focused reasoning |
| grok-4 | Frontier-scale complex reasoning |
| grok-4-fast-reasoning | Accelerated agentic reasoning |
| grok-3 / grok-3-mini | Strong reasoning / lightweight interactive |
| Llama-3.3-70B-Instruct | Enterprise Q&A, decision support |
| Llama-4-Maverick-17B-128E-Instruct-FP8 | Fast, cost-efficient inference |
| DeepSeek-V3-0324 / V3.1 | Multimodal understanding |
| DeepSeek-R1-0528 | Long-form reasoning |
| gpt-oss-120b | Open-ecosystem transparency |

---
# PART 9: HOSTED AGENT PERMISSIONS (RBAC)
---

## 9.1 Permission Classes

Three permission classes for Hosted agents:
1. **User/principal permissions** — working with Foundry resources
2. **Project permissions** — granted to the Foundry project
3. **Agent permissions** — granted to the agent itself

### Key Built-in Roles

| Role | Purpose |
|---|---|
| **Owner** | Full ARM permissions (no data plane) |
| **Contributor** | Create/manage Azure resources (no data plane) |
| **Foundry User** | Create agents, model inference, interact |
| **Foundry Project Manager** | Manage projects + create agents + assign Foundry User role |
| **Foundry Account Owner** | Create deployments, manage projects (no data plane) |
| **Foundry Owner** | Full control plane + data plane (can't create role assignments) |

### Operations & Required Roles

| Operation | Required Permission | Recommended Role |
|---|---|---|
| Create Foundry account | `accounts/write` | Owner / Contributor |
| Deploy a model | `accounts/deployments/write` | Owner / Contributor |
| Create a project | `accounts/projects/write` | Owner / Foundry Project Manager |
| Create an agent | `AIServices/agents/write` (data plane) | **Foundry User** |
| Interact with agent | `applications/invoke/action` (data plane) | **Foundry User** |
| Push image to ACR | ACR data action | Container Registry Repository Writer |
| View telemetry | App Insights read | Monitoring Reader |

### Identity Model
- Every Hosted agent gets a **dedicated Entra agent identity** at deploy time
- The project managed identity handles infrastructure (image pulls)
- Agent identity needs **Foundry User** role on the project for model inference

> **Recommended:** Use **Foundry Project Manager** for agent creators — includes data plane permissions + ability to assign Foundry User role.

---
# PART 10: MCP TOOL GOVERNANCE VIA AI GATEWAY
---

## 10.1 Governing MCP Tools

Route MCP traffic through an **AI Gateway** (Azure API Management) to enforce:
- **Authentication** enforcement
- **Rate limits** (calls per minute)
- **IP filtering** (trusted networks only)
- **Audit logging** with correlation IDs
- **Routing policies** (geographic backends)

### Setup Steps
1. Connect AI Gateway to Foundry resource
2. Add tool via Foundry portal (Tools → Catalog or Custom)
3. Verify endpoint shows gateway URL, not direct MCP server URL
4. Apply API Management policies in Azure portal

### Common Policies

```xml
<!-- Rate Limiting -->
<inbound>
  <base />
  <rate-limit-by-key calls="60" renewal-period="60"
    counter-key="@(context.Request.IpAddress)" />
</inbound>

<!-- IP Filtering -->
<inbound>
  <base />
  <ip-filter action="allow">
    <address>10.0.0.0/24</address>
  </ip-filter>
</inbound>

<!-- Correlation ID for tracing -->
<inbound>
  <base />
  <set-header name="X-Correlation-Id" exists-action="override">
    <value>@(context.RequestId)</value>
  </set-header>
</inbound>

<!-- Geographic Routing -->
<inbound>
  <base />
  <choose>
    <when condition="@(context.Request.Headers.GetValueOrDefault('X-Region','us') == 'eu')">
      <set-backend-service base-url="https://europe-api.contoso-mcp.net" />
    </when>
  </choose>
</inbound>
```

### Limitations
- Only MCP tools supported (not SharePoint, OpenAPI, etc.)
- Gateway routing applied only at tool creation time
- Tools with managed OAuth not supported
- Policies managed via Azure portal only (not Foundry portal)

---
# PART 11: MIGRATION GUIDES
---

## 11.1 Migration from Assistants API → Foundry Agent Service

### Key Concept Mapping

| Old (Assistants API) | New (Foundry Agent Service) |
|---|---|
| Assistants | Agents (with versioning) |
| Threads | Conversations |
| Runs | Responses |
| Messages | Conversation items |
| `openai` package | `azure-ai-projects>=2.0.0` |

### SDK Migration

```python
# OLD: Assistants API
# from openai import AzureOpenAI
# client = AzureOpenAI(azure_endpoint=..., api_key=...)
# assistant = client.beta.assistants.create(model="gpt-4o", ...)
# thread = client.beta.threads.create()
# run = client.beta.threads.runs.create(thread_id=..., assistant_id=...)

# NEW: Foundry Agent Service
# from azure.ai.projects import AIProjectClient
# from azure.identity import DefaultAzureCredential
# project = AIProjectClient(endpoint=..., credential=DefaultAzureCredential())
# agent = project.agents.create_version(agent_name=..., definition=...)
# openai = project.get_openai_client()
# conversation = openai.conversations.create()
# response = openai.responses.create(conversation=..., input=...)
```

### Migration Tool
A [migration tool](https://aka.ms/agent/migrate/tool) is available to help automate the process.

## 11.2 Hosted Agent Preview Migration

The refreshed preview introduces major changes:

### Key Changes

| Aspect | Initial Preview | Refreshed Preview |
|---|---|---|
| **Compute** | Manual start/stop/replicas | Automatic lifecycle (15min idle timeout) |
| **Isolation** | Shared | Session-based sandbox |
| **Packages** | Framework adapters (`agentserver-agentframework`) | Protocol libraries (`agentserver-responses`) |
| **Identity** | Shared project managed identity | Dedicated Entra identity per agent |
| **Endpoint** | Shared project endpoint + `agent_reference` | Dedicated per-agent endpoint |
| **Protocols** | Responses only | Responses + Invocations + Activity + A2A |
| **Protocol Version** | `"v1"` | `"1.0.0"` (semver) |

### Package Migration

| Old Package | New Package |
|---|---|
| `azure-ai-agentserver-agentframework` | Removed → use `agent-framework-foundry-hosting` |
| `azure-ai-agentserver-langgraph` | Removed → use `azure-ai-agentserver-responses` |
| `azure-ai-projects>=2.0.0` | `azure-ai-projects>=2.1.0` |

### SDK Method Changes

```python
# OLD: Shared endpoint with extra_body
# openai = project.get_openai_client()
# response = openai.responses.create(
#     input="Hello!",
#     extra_body={"agent_reference": {"name": "my-agent", "type": "agent_reference"}}
# )

# NEW: Dedicated agent endpoint
# openai = project.get_openai_client(agent_name="my-agent")
# response = openai.responses.create(input="Hello!")
```

> **Deadline:** Initial preview backend supported until **May 22, 2026**. Redeploy required.

## 11.3 Agent Applications Migration

The new agent object model collapses **Agent Applications** and **Agent Deployments** into the **Agent** object.

### Before vs. After

| Aspect | Legacy Model | New Model |
|---|---|---|
| **Resources** | Agent + Application + Deployment (separate) | Agent only (unified) |
| **Identity** | Shared → unique only at publish time | Unique Entra identity from creation |
| **Endpoint** | Created via Agent Application | Built-in `agent_endpoint` |
| **Publishing** | Two-step: create app + publish to M365 | One-step: publish to M365/Teams |

### Migration Paths

| Path | Action |
|---|---|
| **New agents** | No action — automatically get new model |
| **Legacy agents** (`identity == null`) | Create new agent with same definition |
| **Existing Agent Applications** | Create new agent → publish to M365 → decommission old Application |

### Endpoint URL Changes

| Type | Legacy | New |
|---|---|---|
| Responses | `.../applications/{app}/protocols/openai` | `.../agents/{agent}/endpoint/protocols/openai/v1/responses` |
| Activity | `.../applications/{app}/protocols/activityprotocol` | `.../agents/{agent}/endpoint/protocols/activityprotocol` |

---
# PART 12: FREQUENTLY ASKED QUESTIONS
---

## 12.1 Key FAQs

### Setup

**Q: What's the difference between basic and standard setup?**
- **Basic:** Agent state stored in Microsoft-managed resources
- **Standard:** Agent data stored in YOUR Azure resources via capability hosts

**Q: What permissions do I need?**
- Foundry Account Owner → create accounts/projects
- Foundry User → create and edit agents
- RBAC Admin or Owner → standard setup role assignments

### Data

**Q: Does the service store data?**
Yes — conversations, responses, files, and vector stores. Stateful API.

**Q: Where is data stored?**
- Basic: Microsoft-managed, logically separated storage
- Standard: Your own Azure Storage, Cosmos DB, AI Search

**Q: Does Microsoft use my data for training?**
**No.**

**Q: Does it support CMK encryption?**
- Basic: Microsoft-managed keys only
- Standard: Customer-managed keys (CMK) supported

### Pricing

**Q: How am I charged?**
- Inference cost (input/output) of the base model per agent
- Code Interpreter: charged per session (default 1hr active)
- File search: billed on vector storage used

### Networking

**Q: What IP ranges are supported for agent subnets?**
- Class A: 10.0.0.0/8
- Class B: 172.16.0.0/12
- Class C: 192.168.0.0/16
(Public IP ranges NOT supported)

**Q: Minimum agent subnet size?**
Recommended /24, minimum /27.

**Q: Can multiple Foundry resources share a VNet?**
Yes, but each needs its own dedicated agent runtime subnet.

---
# PART 13: BEST PRACTICES & SECURITY CHECKLIST
---

## 13.1 Tracing Best Practices

- **Use consistent span attributes** across all agents and tools
- **Correlate evaluation run IDs** with trace data for unified analysis
- **Redact sensitive content** from prompts, tool arguments, and span attributes
- **Disable content recording in production** unless compliance allows it
- **Don't store secrets** in prompts, tool arguments, or span attributes

## 13.2 Monitoring Best Practices

- Set up **continuous evaluation** with appropriate sample rates
- Configure **alerts** for latency >10s and success rate <95%
- Monitor **token usage trends** to optimize costs
- Enable **red team scans** for security-sensitive agents
- Use **evaluation as a CI/CD quality gate** before deployment

## 13.3 Limits Best Practices

- **Keep files small and focused** — multiple small > one large
- **Avoid very large messages** — put long content in files + file search
- **Rotate threads** for long conversations
- **Register only required tools** (max 128)
- **Implement exponential backoff** for rate limit (429) errors

## 13.4 Production Security Checklist

| Check | Description |
|---|---|
| [ ] | Content recording disabled in production |
| [ ] | Baggage propagation disabled (default) |
| [ ] | No secrets in prompts or span attributes |
| [ ] | Log Analytics Reader role assigned |
| [ ] | Least-privilege RBAC for all identities |
| [ ] | Agent identity has Foundry User role |
| [ ] | MCP tools routed through AI Gateway |
| [ ] | Continuous evaluation enabled |
| [ ] | Capability hosts configured for production |
| [ ] | VNet isolation for sensitive workloads |

---
# PART 14: TROUBLESHOOTING REFERENCE
---

## 14.1 Common Issues

### Tracing Issues

| Issue | Cause | Fix |
|---|---|---|
| No traces in portal | Not connected / no traffic | Connect App Insights, generate traffic, wait 2-5 min |
| Authorization errors | Missing RBAC | Assign Log Analytics Reader role |
| Client traces missing | Instrumentation not set up | Check `AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING=true` |
| Content not in spans | Content recording off | Set `OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT=true` |
| Client/server spans not correlated | Propagation disabled | Enable trace context propagation, use `get_openai_client()` |

### Monitoring Issues

| Issue | Cause | Fix |
|---|---|---|
| Dashboard charts empty | No traffic / time range wrong | Generate traffic, expand time range |
| Continuous eval missing | Rule not enabled | Verify rule + project MI has Foundry User role |
| Eval runs skipped | Hourly limit reached | Increase `max_hourly_runs` |

### Capability Host Issues

| Issue | Cause | Fix |
|---|---|---|
| 409 Conflict | Already exists at scope | Use GET to check, use same name |
| Concurrent ops conflict | Another operation in progress | Wait + retry with exponential backoff |

---
## Summary

This notebook covered the complete production operations landscape for Foundry agents:

1. **Tracing & Observability** — OpenTelemetry-based tracing with Application Insights, client-side instrumentation, custom spans
2. **Framework Integrations** — Native tracing for Agent Framework, LangChain/LangGraph, and OpenAI Agents SDK
3. **Prompt Optimizer** — AI-driven improvement of agent instructions
4. **Monitoring Dashboard** — Metrics, continuous evaluation, red teaming, and alerts
5. **Agent Evaluation** — Built-in evaluators for quality, safety, and agent behavior
6. **Agent 365 Integration** — Enterprise governance via registry, access control, and security
7. **Capability Hosts** — Bring-your-own-resources for data sovereignty
8. **Quotas & Limits** — Service boundaries and error handling
9. **Permissions (RBAC)** — Comprehensive role model for hosted agents
10. **MCP Tool Governance** — AI Gateway for rate limiting, IP filtering, audit logging
11. **Migration Paths** — From Assistants API, hosted preview, and agent applications

**Next Steps:**
- Set up Application Insights for your Foundry project
- Enable client-side tracing in your agent application
- Configure continuous evaluation for production agents
- Plan migration from legacy APIs if applicable
- Review RBAC assignments for least-privilege access